# 人工介入流程：預行動閘門、風險分級與審計日誌

本課程的 README 介紹了人工介入流程，使用一段簡短的程式碼片段，要求用戶在代理程式產生回應後「批准」或「拒絕」。這種模式是一個不錯的起點，但實際的 HITL 實作通常需要三個額外元素：

1. 一個 <strong>預行動閘門</strong>，用於在代理程式執行風險步驟 <strong>之前</strong> 運行，以便控制成本、不可逆性與延遲。
2. <strong>風險分級</strong>，使低風險動作自動執行、中等風險動作批次批准，只有高風險動作需阻塞並由人工介入。
3. 一個 <strong>審計日誌加修訂循環</strong>，記錄每次閘門決策為 JSONL 格式，並且拒絕時透過結構化理由重新提示代理，而不只是顯示「Revising...」。

本筆記本基於 `06-system-message-framework.ipynb` 中相同的原語搭建上述機制。它可在 `DEMO_MODE = True` （無需互動輸入）下端到端運行，或在 `DEMO_MODE = False` 時以真實 `input()` 提示輸入。注意：在 DEMO_MODE 下第三步的重試會以腳本方式執行，讓流程機制完整可見。真正的修訂驅動重分類需要 `DEMO_MODE = False` 並且有操作人員協助。

**非本課程範圍（由其他課程處理）：** 認證與存取控制（第06課 README 中的威脅2）、工具呼叫中介軟體（第14課 MAF 深入探討）、多代理辯論模式。


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## 模式 1：前置動作閘門

README 的 HITL 範例先調用代理，然後再請使用者批准輸出。那是一個<strong>後置動作</strong>流程。代理已經執行，因此 LLM 呼叫成本已經支付，且任何副作用（寄送郵件、寫入資料庫列、發表評論）已經發生。

<strong>前置動作</strong>流程則會在代理執行風險步驟之前插入閘門。代理提出動作，閘門決定是否執行，只有在批准後副作用才會發生。

| 方面 | 後置動作批准（README 範例） | 前置動作閘門（本筆記本） |
|---|---|---|
| 什麼時候執行批准？ | 代理產出結果之後 | 任何副作用執行之前 |
| 拒絕時的 LLM 成本 | 已支付 | 僅支付提案成本，非動作成本 |
| 無法回復的副作用 | 可能（動作已發生） | 已預防 |
| 審計清晰度 | 批准是一個列印語句 | 批准是帶有時間戳、動作和理由的 JSON 紀錄 |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## 模式 2：風險分級

並非所有操作都需要人類批准。對公共 API 進行只讀查詢的風險與發送客戶電郵的風險不同。將兩者同等對待會浪費操作員的注意力，並減慢代理的反應速度。

一個簡單的三層模型：

| 層級 | 範例 | 批准流程 |
|---|---|---|
| `低`（只讀） | 搜尋知識庫、查找航班選項、抓取公共網頁 | 自動執行，並記錄以供審計 |
| `中`（小額變更） | 快取結果、增加計數器、安排提醒 | 自動執行，但每日批次審查 |
| `高`（對外或不可逆） | 發送電郵、收取信用卡費用、發布至公共頻道 | 阻擋，需人工批准 |

這是一種分級方式。生產系統通常使用更細緻的分級（例如，AWS IAM 權限層級、基於角色的存取層級）。以下的三層版本是對於混合只讀和副作用動作的代理來說，最小且有用的版本。

以下分類器使用關鍵字啟發式，讓示範保持確定性且成本低。在生產系統中，你會用訓練好的分類器或政策引擎取代它。


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## 模式 3：審核紀錄與修訂循環

`print("Response approved.")` 並不是審核紀錄。為了建立信任，每個閘門決策都應該被記錄為結構化事件，讓你之後能查詢、重播，或附加到事件審查中。

兩個部分：

1. **追加式 JSONL。** 每個決策一行，包含時間戳、行動、層級、決策、原因。方便用 grep 搜尋，也方便之後送到真正的日誌儲存。
2. **拒絕時的修訂循環。** 當閘門回傳 `deny` 時，代理會在上下文中帶入拒絕原因，重新提示自己，讓下一次的提案能避免同樣問題。


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## 額外資源

有其他數個公開專案實作了這些 HITL 模式的變體。比較不同方法以找出最適合你的技術棧：

- **LangChain** 人類在回路中的工具包裝器 ([docs](https://python.langchain.com/docs/integrations/tools/human_tools))：即插即用的工具包裝器，可在執行時暫停以待人工輸入。
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ 重新架構)：使用代理角色專門代表多代理對話中的人類。
- **Microsoft Agent Framework (MAF)** 函數調用中介軟體 ([docs](https://learn.microsoft.com/agent-framework/))：環繞每個工具／函數呼叫的中介軟體，適合用於閘道邏輯和審批流程。

各專案對三個子模式的處理方式不同：LangChain 將它們包裝成工具、AutoGen 使用代理角色、而 Microsoft Agent Framework 使用函數調用中介軟體。至少完整閱讀一兩個實作，再決定你自己的代理設計。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
